In [20]:
import torch
from PIL import Image
import torch.nn as nn
import os
import numpy as np
from torchvision.transforms import transforms

In [2]:
class AlexNetCifar10(nn.Module):
  def __init__(self,num_classes=10):
    super(AlexNetCifar10,self).__init__()
    self.features=nn.Sequential(
        #(3x32,32)->(64,32,32)
        nn.Conv2d(in_channels=3,out_channels=64,kernel_size=3,stride=1,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),#(64,16,16)

        nn.Conv2d(in_channels=64,out_channels=196,kernel_size=3,stride=1,padding=1),#(64,16,16)->(196,16,16)
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),#(196,8,8)

        #(196,8,8)->(256,8,8)
        nn.Conv2d(in_channels=196,out_channels=256,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),

        #(256,8,8)->(256,8,8)
        nn.Conv2d(in_channels=256,out_channels=256,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2) #(256,4,4)
    )
    self.classifier=nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(256*4*4,4096),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(4096,4096),
        nn.ReLU(0.5),
        nn.Linear(4096,num_classes)
    )
  def forward(self,x):
    x=self.features(x)
    x=torch.flatten(x,1)
    x=self.classifier(x)

    return x

In [13]:
def predict_new(model,image_path,label_names,device):
  image=Image.open(image_path).convert('RGB')
  transform=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.6,0.4],[0.5,0.6,0.4])

])
  image_tensor=transform(image).to(device)
  image_tensor=image_tensor.unsqueeze(0)
  model.eval()
  with torch.no_grad():
    pred =model(image_tensor)
    pred_class=torch.argmax(pred).item()
  name=label_names[pred_class]

  return f"The item in the Picture is '{ name }' predicted by Model."


In [14]:
labels_names= ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [7]:
device='cpu'


In [8]:
model=AlexNetCifar10().to(device)

In [10]:
state_dict=torch.load('AlexNet.pth',map_location=torch.device(device),weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [18]:
image_dir='Real'
img1='plane.jfif'
image1=os.path.join(image_dir,img1)

In [21]:
predict_new(model,image_path=image1,label_names=labels_names,device=device)

"The item in the Picture is 'airplane' predicted by Model."

In [22]:
img2='car.jfif'
image2=os.path.join(image_dir,img2)
predict_new(model,image_path=image2,label_names=labels_names,device=device)

"The item in the Picture is 'automobile' predicted by Model."

In [23]:
img3='bird.jfif'
image3=os.path.join(image_dir,img3)
predict_new(model,image_path=image3,label_names=labels_names,device=device)

"The item in the Picture is 'bird' predicted by Model."

In [24]:
img4='images.jfif'
image4=os.path.join(image_dir,img4)
predict_new(model,image_path=image4,label_names=labels_names,device=device)

"The item in the Picture is 'airplane' predicted by Model."

In [25]:
img5='truck.jfif'
image5=os.path.join(image_dir,img5)
predict_new(model,image_path=image5,label_names=labels_names,device=device)

"The item in the Picture is 'ship' predicted by Model."

In [26]:
img6='ship.jfif'
image6=os.path.join(image_dir,img6)
predict_new(model,image_path=image6,label_names=labels_names,device=device)

"The item in the Picture is 'ship' predicted by Model."